# Supervised 2D TEM Training, Loading, and Latent Visualization

This notebook runs the supervised 4x4 grid training loop on Colab, reloads the trained checkpoint, extracts node-level `g` and `p` vectors, saves the `p` tensor, and visualizes both latent spaces.

In [ ]:
import importlib
import subprocess
import sys

def ensure_package(package_name):
    try:
        importlib.import_module(package_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

for package in ["scipy", "tensorboard", "matplotlib"]:
    ensure_package(package)

print("Dependencies ready.")

In [ ]:
import os

repo_dir = "/content/torch_tem"
if not os.path.exists(repo_dir):
    !git clone https://github.com/YifeiCAO/torch_tem.git {repo_dir}
else:
    print(f"Repository already exists at {repo_dir}")
    !git -C {repo_dir} pull

%cd /content/torch_tem

In [ ]:
import importlib
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from dataset_wine_2d import Wine2DDataset
import train_2d_sl
importlib.reload(train_2d_sl)

train = train_2d_sl.train
load_supervised_model = train_2d_sl.load_supervised_model

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("train_2d_sl path:", train_2d_sl.__file__)

In [ ]:
NUM_EPOCHS = 20
NUM_SAMPLES = 4096
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

CHECKPOINT_PATH = "/content/torch_tem/supervised_tem_2d.pt"
P_VECTORS_PATH = "/content/torch_tem/p_vectors_4x4.pt"
G_VECTORS_PATH = "/content/torch_tem/g_vectors_4x4.pt"

In [ ]:
dataset = Wine2DDataset(num_samples=16)
loader = DataLoader(dataset, batch_size=8, shuffle=True)
current_state, action, next_state = next(iter(loader))

print("current_state:", current_state)
print("action:", action)
print("next_state:", next_state)

In [ ]:
supervised_model = train(
    num_epochs=NUM_EPOCHS,
    num_samples=NUM_SAMPLES,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
)

torch.save(supervised_model.state_dict(), CHECKPOINT_PATH)
print(f"Saved checkpoint to {CHECKPOINT_PATH}")

In [ ]:
loaded_model = load_supervised_model(
    checkpoint_path=CHECKPOINT_PATH,
    batch_size=BATCH_SIZE,
    num_states=16,
    device="cpu",
)

loaded_model.eval()
tem = loaded_model.tem
print("Checkpoint loaded and model set to eval mode.")

In [ ]:
with torch.no_grad():
    node_ids = torch.arange(16, dtype=torch.long)

    g_flat = loaded_model.state_embedding(node_ids)
    g_modules = loaded_model._split_g(g_flat)

    x = torch.eye(tem.hyper["n_x"], dtype=torch.float)[:16]
    x_c = tem.f_c(x)
    x_prev = [torch.zeros((16, tem.hyper["n_x_f"][f]), dtype=torch.float) for f in range(tem.hyper["n_f"])]
    x_f = tem.x_prev2x(x_prev, x_c)
    x_ = tem.x2x_(x_f)

    g_memory = tem.g2g_(g_modules)
    p_modules = tem.inf_p(x_, g_memory)

    g_vectors = torch.cat(g_modules, dim=1)
    p_vectors = torch.cat(p_modules, dim=1)

torch.save(g_vectors, G_VECTORS_PATH)
torch.save(p_vectors, P_VECTORS_PATH)

print("g_vectors shape:", tuple(g_vectors.shape))
print("p_vectors shape:", tuple(p_vectors.shape))
print(f"Saved g vectors to {G_VECTORS_PATH}")
print(f"Saved p vectors to {P_VECTORS_PATH}")

In [ ]:
state_id = 0
print("g[0][:20] =", g_vectors[state_id, :20])
print("p[0][:20] =", p_vectors[state_id, :20])

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), constrained_layout=True)

im0 = axes[0].imshow(g_vectors.numpy(), aspect="auto", cmap="viridis")
axes[0].set_title("g vectors across 16 grid nodes")
axes[0].set_ylabel("node index")
axes[0].set_xlabel("g dimension")
fig.colorbar(im0, ax=axes[0], fraction=0.02, pad=0.02)

im1 = axes[1].imshow(p_vectors.numpy(), aspect="auto", cmap="magma")
axes[1].set_title("p vectors across 16 grid nodes")
axes[1].set_ylabel("node index")
axes[1].set_xlabel("p dimension")
fig.colorbar(im1, ax=axes[1], fraction=0.02, pad=0.02)

plt.show()

In [ ]:
def project_to_2d(vectors):
    centered = vectors - vectors.mean(dim=0, keepdim=True)
    U, S, V = torch.pca_lowrank(centered, q=2)
    return centered @ V[:, :2]

g_2d = project_to_2d(g_vectors)
p_2d = project_to_2d(p_vectors)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

axes[0].scatter(g_2d[:, 0].numpy(), g_2d[:, 1].numpy(), s=60)
for i in range(16):
    axes[0].text(g_2d[i, 0].item(), g_2d[i, 1].item(), str(i))
axes[0].set_title("PCA of g vectors")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

axes[1].scatter(p_2d[:, 0].numpy(), p_2d[:, 1].numpy(), s=60)
for i in range(16):
    axes[1].text(p_2d[i, 0].item(), p_2d[i, 1].item(), str(i))
axes[1].set_title("PCA of p vectors")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")

plt.show()

## Notes

- `g_vectors` here come from the learned supervised state embedding.
- `p_vectors` are built using the TEM internal pathway names: `x -> f_c -> x_prev2x -> x2x_`, `g -> g2g_`, then `inf_p(x_, g_)`.
- Each row corresponds to one node in the 4x4 grid.